In [25]:
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, UMT5Config, UMT5ForConditionalGeneration, Seq2SeqTrainingArguments,Seq2SeqTrainer, EarlyStoppingCallback 
import torch
from datasets import Dataset, DatasetDict
import transformers, dataclasses
from tqdm.auto import tqdm
import difflib

---
## Load Dataset

In [26]:
def load_jsonl(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f]
    return pd.DataFrame(records)

train_df = load_jsonl("../data/processed/train.jsonl")
val_df = load_jsonl("../data/processed/val.jsonl")
test_real_df = load_jsonl("../data/processed/test_real.jsonl")
test_regression_df = load_jsonl("../data/processed/test_regression.jsonl")

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test_real": Dataset.from_pandas(test_real_df),
    "test_regression": Dataset.from_pandas(test_regression_df),
})

In [27]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 168326 entries, 0 to 168325
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   corrupted  168326 non-null  str  
 1   clean      168326 non-null  str  
 2   source     168326 non-null  str  
dtypes: str(3)
memory usage: 85.1 MB


---
## Tokenizer and model

In [28]:
MODEL_NAME = "google/umt5-small"

In [29]:
MAX_LENGTH = 128 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    model_inputs = tokenizer(
        examples["corrupted"],
        max_length=MAX_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples["clean"],
        max_length=MAX_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [30]:
config = UMT5Config.from_pretrained(MODEL_NAME)
config.tie_word_embeddings = False          # umt5 untied — сохраняем обученный lm_head при загрузке
model = UMT5ForConditionalGeneration.from_pretrained(MODEL_NAME, config=config)

In [31]:
def unify_columns(ds):
    rename_map = {}
    if "input" in ds.column_names:
        rename_map["input"] = "corrupted"
    if "target" in ds.column_names:
        rename_map["target"] = "clean"
    return ds.rename_columns(rename_map) if rename_map else ds

dataset = DatasetDict({split: unify_columns(ds) for split, ds in dataset.items()})

In [32]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
)

In [33]:
tokenized_datasets = DatasetDict({
    split: ds.map(preprocess, batched=True, remove_columns=ds.column_names)
    for split, ds in dataset.items()
})

print(tokenized_datasets)
print(tokenized_datasets["train"][0])

Map: 100%|██████████| 105/105 [00:00<00:00, 14996.49 examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 168326
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2400
    })
    test_real: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 120
    })
    test_regression: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 105
    })
})
{'input_ids': [297, 110099, 20944, 273, 284, 280, 1174, 90123, 1109, 105042, 463, 16970, 463, 1174, 292, 914, 296, 150299, 274, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [297, 110099, 20944, 273, 284, 280, 1174, 90123, 1109, 76954, 10056, 54745, 73180, 274, 1]}


---

## Training

In [22]:
training_args = Seq2SeqTrainingArguments(
    output_dir="../models/umt5-gec-small",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,      
    bf16=True,
    optim="adafactor",
    num_train_epochs=1,
    learning_rate=3e-4,
    warmup_steps=500,
    lr_scheduler_type="linear",
    group_by_length=True,               
    eval_strategy="steps", eval_steps=1000,
    save_strategy="steps", save_steps=1000,
    save_total_limit=2, logging_steps=50,
    predict_with_generate=False,        
    generation_max_length=MAX_LENGTH,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

In [23]:
model.config.use_cache = False

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()


trainer.save_model("../models/umt5-gec-small/best")
tokenizer.save_pretrained("../models/umt5-gec-small/best")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Step,Training Loss,Validation Loss
1000,0.630000,0.388591
2000,0.487600,0.284214
3000,0.412900,0.231090
4000,0.389200,0.213168
5000,0.369000,0.206780


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


('../models/umt5-gec-fixed/best\\tokenizer_config.json',
 '../models/umt5-gec-fixed/best\\special_tokens_map.json',
 '../models/umt5-gec-fixed/best\\spiece.model',
 '../models/umt5-gec-fixed/best\\added_tokens.json',
 '../models/umt5-gec-fixed/best\\tokenizer.json')

## Evaluation

In [ ]:
# Evaluate the SAVED checkpoint (fresh load), not the in-memory `model` —
# load_best_model_at_end reloads best with missing embed_tokens, so `model`
# generates garbage; the on-disk checkpoint loads correctly.
import json, torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

CKPT = "../models/umt5-gec-small/best"
eval_tok = AutoTokenizer.from_pretrained(CKPT)
eval_model = AutoModelForSeq2SeqLM.from_pretrained(CKPT).to(
    "cuda" if torch.cuda.is_available() else "cpu").eval()
eval_model.config.use_cache = True


@torch.no_grad()
def correct_batch(texts, bs=16):
    out = []
    for i in range(0, len(texts), bs):
        b = texts[i:i + bs]
        x = eval_tok(b, return_tensors="pt", truncation=True,
                     max_length=MAX_LENGTH, padding=True).to(eval_model.device)
        y = eval_model.generate(x.input_ids, attention_mask=x.attention_mask,
                                max_new_tokens=MAX_LENGTH, num_beams=4,
                                no_repeat_ngram_size=3, early_stopping=True)
        out += eval_tok.batch_decode(y, skip_special_tokens=True)
    return out


def load_eval_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8")]


def char_acc(a, b):
    a, b = a.strip(), b.strip()
    return sum(x == y for x, y in zip(a, b)) / max(len(a), len(b), 1)


def levenshtein(a, b):
    a, b = a.strip(), b.strip()
    d = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(b) + 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
    return d[len(b)]


def word_acc(p, t):
    pw, tw = p.split(), t.split()
    return sum(x == y for x, y in zip(pw, tw)) / max(len(tw), 1)

In [ ]:
real = load_eval_jsonl("../data/processed/test_real.jsonl")
src = [r["corrupted"] for r in real]
tgt = [r["clean"] for r in real]
pred = correct_batch(src)

# char-level
before = sum(char_acc(s, t) for s, t in zip(src, tgt)) / len(real)
after  = sum(char_acc(p, t) for p, t in zip(pred, tgt)) / len(real)
exact  = sum(p.strip() == t.strip() for p, t in zip(pred, tgt)) / len(real)

# edit-distance & word-level
d_src  = [levenshtein(s, t) for s, t in zip(src, tgt)]
d_pred = [levenshtein(p, t) for p, t in zip(pred, tgt)]
avg_src, avg_pred = sum(d_src) / len(real), sum(d_pred) / len(real)
improved = sum(dp < ds for dp, ds in zip(d_pred, d_src))
worse    = sum(dp > ds for dp, ds in zip(d_pred, d_src))
wacc_before = sum(word_acc(s, t) for s, t in zip(src, tgt)) / len(real)
wacc_after  = sum(word_acc(p, t) for p, t in zip(pred, tgt)) / len(real)

print("=== test_real (n =", len(real), ") ===")
print(f"char-acc     : {before:.3f} -> {after:.3f}")
print(f"word-acc     : {wacc_before:.3f} -> {wacc_after:.3f}")
print(f"exact-match  : {exact:.3f}")
print(f"edit-dist    : {avg_src:.2f} -> {avg_pred:.2f}  (error reduction {1 - avg_pred/avg_src:.1%})")
print(f"improved     : {improved}/{len(real)} ({improved/len(real):.1%})")
print(f"worse        : {worse}/{len(real)} ({worse/len(real):.1%})")

print("\n--- samples ---")
for r, p in list(zip(real, pred))[:5]:
    print("SRC :", r["corrupted"])
    print("TGT :", r["clean"])
    print("PRED:", p, "\n")

In [ ]:
from collections import defaultdict

reg = load_eval_jsonl("../data/processed/test_regression.jsonl")
rsrc = [r["input"] for r in reg]
rpred = correct_batch(rsrc)

unchanged = sum(p.strip() == s.strip() for s, p in zip(rsrc, rpred)) / len(reg)
by = defaultdict(list)
for r, s, p in zip(reg, rsrc, rpred):
    by[r.get("type", "?")].append(p.strip() == s.strip())

print("=== test_regression (n =", len(reg), ") ===")
print(f"clean text left UNCHANGED: {unchanged:.3f}   (higher = less over-correction)")
for t, v in by.items():
    print(f"  type={t}: {sum(v)/len(v):.3f} ({len(v)} ex)")